In [2]:
from pathlib import Path
import qrcode

links = {
    "pohsuan": "https://peh-suan.github.io/#home",
    "canaan": "https://www.cbreiss.com",
}

output_dir = Path("/Users/pohsuan/Desktop/")
output_dir.mkdir(exist_ok=True)

for name, url in links.items():
    qr = qrcode.QRCode(
        error_correction=qrcode.constants.ERROR_CORRECT_M,
        box_size=20,
        border=4,
    )
    qr.add_data(url)
    qr.make(fit=True)

    image = qr.make_image(fill_color="#011F5B", back_color="white")
    image.save(output_dir / f"{name}.png")

In [2]:
import pandas as pd
import tqdm, os
import pickle
import warnings
warnings.simplefilter("ignore")
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Subset

import sys
sys.path.insert(0, "../..")

from utils.scaling import fit_scalers, apply_scaler
from utils.data_processing import dataset_df_to_token_df
from utils.acoustics_processing import get_mfccs
from utils.articulation_processing import get_aeema_articulatory_features

In [3]:
df = pd.read_csv("../data_processing/dataset_construction/datasets/processed_aeema.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[2])
token_df = dataset_df_to_token_df(df)

In [4]:
with open(f"../encoder_training/scalers/aeema_art.pkl", "rb") as f:
    art_scalers = pickle.load(f)
with open(f"../encoder_training/scalers/aeema_ac.pkl", "rb") as f:
    ac_scalers = pickle.load(f)

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
from utils.models import MDN
import copy

from utils.models import SequenceAutoencoder

device = "mps" if torch.backends.mps.is_available() else "cpu"

art_model = SequenceAutoencoder(
    feature_shape=(6, 3),
    latent_dim=64,
).to(device)
ckpt = torch.load(
    "../encoder_training/ckpts/aeema/art_model_ckpt.pth",
    map_location=device,
)
art_model.load_state_dict(ckpt["model"])
art_model.eval()

ac_model = SequenceAutoencoder(
    feature_shape=13,
    latent_dim=64,
).to(device)
ac_model.load_state_dict(torch.load("../encoder_training/ckpts/aeema/ac_model_ckpt.pth", map_location=device))
ac_model.eval()

SequenceAutoencoder(
  (encoder_cnn): TimeMargin(
    (body): Sequential(
      (0): Conv1d(13, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (1): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (2): ReLU()
      (3): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (4): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (5): ReLU()
      (6): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (7): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (8): ReLU()
    )
  )
  (encoder_proj): Linear(in_features=128, out_features=256, bias=True)
  (trunk): TemporalTrunk(
    (encoder_gru): GRU(256, 256, batch_first=True, bidirectional=True)
    (pool): MaskedPool(
      (score): Sequential(
        (0): Linear(in_features=512, out_features=256, bias=True)
        (1): Tanh()
        (2): Linear(in_features=256, out_features=1, bias=True)
      )
    )
   

In [6]:
for name, param in art_model.named_parameters():
    if not torch.isfinite(param).all():
        print(
            name,
            "NaN:", torch.isnan(param).sum().item(),
            "Inf:", torch.isinf(param).sum().item(),
        )

In [7]:
art_vectors = []
ac_vectors = []
phonemes = []
subjects = []

pbar = tqdm.tqdm(token_df.iterrows(), total=len(token_df))

batch = []
batch_idx = 0
for idx, (_, data) in enumerate(pbar):

    start, end, phone, art_f, ac_f, subject = data["start"], data["end"], data["phone"], data["art_f"], data["ac_f"], data["subject"]

    articulatory_features = get_aeema_articulatory_features(art_f, start, end)
    if articulatory_features is None: continue
    articulatory_features = articulatory_features.reshape(articulatory_features.shape[0], 6, 3)
    articulatory_features = apply_scaler(articulatory_features, art_scalers[subject])
    articulatory_features = torch.from_numpy(articulatory_features).float().unsqueeze(0).to(device)
    art_out, art_z = art_model(articulatory_features, return_latent=True)

    mfccs = get_mfccs(ac_f, start, end)
    if mfccs is None: continue
    mfccs = apply_scaler(mfccs, ac_scalers[subject])
    mfccs = torch.from_numpy(mfccs).unsqueeze(0).to(device)
    ac_out, ac_z = ac_model(mfccs, return_latent=True)

    art_vectors.append(art_z.detach().cpu().numpy().squeeze())
    ac_vectors.append(ac_z.detach().cpu().numpy().squeeze())
    phonemes.append(phone)
    subjects.append(subject)

100%|██████████| 56882/56882 [1:05:28<00:00, 14.48it/s]


In [8]:
import numpy as np
art = np.stack(art_vectors, axis=0)   # (N, T_A, D_A)
ac = np.stack(ac_vectors, axis=0)  # (N, T_X, D_X)

# A: (N, T_A, D_A)
# X: (N, T_X, D_X)
N, D = art.shape
N2, D_X = ac.shape
assert N == N2

art_flat = art.reshape(N, D)
ac_flat = ac.reshape(N, D_X)   # shape (N, x_dim)

print("art_flat:", art_flat.shape, "ac_flat:", ac_flat.shape)

art_flat: (56810, 64) ac_flat: (56810, 64)


In [9]:
art_raw = art_flat.astype(np.float32)
ac_raw = ac_flat.astype(np.float32)

N = len(ac_raw)
k = 5

indices = np.arange(N)
np.random.shuffle(indices)
folds = np.array_split(indices, k)

In [10]:
art_dim = art_raw.shape[1]
ac_dim = ac_raw.shape[1]

In [11]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

# ----------------------------------------------------
# Dataset
# ----------------------------------------------------
class AXDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.from_numpy(X).float()
        self.Y = torch.from_numpy(Y).float()
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

# ----------------------------------------------------
# Training loop for one model
# ----------------------------------------------------
def train_model(train_loader, val_loader, x_dim, y_dim):
    model = MDN(x_dim=x_dim, y_dim=y_dim, n_components=5, hidden_dim=256).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    best_val = float('inf')
    patience = 10
    counter = 0

    for epoch in range(1, 2001):  # just large upper bound
        # ----- train -----
        model.train()
        train_loss = 0
        for Xb, Yb in train_loader:
            Xb, Yb = Xb.to(device), Yb.to(device)
            logp = model.log_prob(Xb, Yb)
            loss = -logp.mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * len(Xb)
        train_loss /= len(train_loader.dataset)

        # ----- validation -----
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for Xb, Yb in val_loader:
                Xb, Yb = Xb.to(device), Yb.to(device)
                loss = -model.log_prob(Xb, Yb).mean()
                val_loss += loss.item() * len(Xb)
        val_loss /= len(val_loader.dataset)

        print(f"Epoch {epoch}: train={train_loss:.4f}, val={val_loss:.4f}")

        # Early stopping check
        if val_loss < best_val - 1e-4:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            counter = 0
            print("   ✓ New best model")
        else:
            counter += 1
            print(f"   No improvement ({counter} epochs)")

        if counter >= patience:
            print(f"Early stopping. Best val={best_val:.4f}")
            break

    model.load_state_dict(best_state)
    return model

In [12]:
logp_full = np.zeros(N)

for fold in range(k):

    print(f"\n==============================")
    print(f"   FOLD {fold}")
    print(f"==============================")

    test_idx = folds[fold]
    train_idx = np.hstack([
        folds[i] for i in range(k) if i != fold
    ])

    # ----------------------------------------
    # 1. Make internal train/validation split
    # ----------------------------------------

    train_idx_shuffled = train_idx.copy()
    np.random.shuffle(train_idx_shuffled)

    split = int(0.9 * len(train_idx_shuffled))

    train_inner = train_idx_shuffled[:split]
    val_inner = train_idx_shuffled[split:]


    # ----------------------------------------
    # 2. Fit scalers ONLY on outer training set
    # ----------------------------------------

    sc_ac = StandardScaler()
    sc_art = StandardScaler()

    sc_ac.fit(ac_raw[train_inner])
    sc_art.fit(art_raw[train_inner])

    # ----------------------------------------
    # 3. Transform all data using these scalers
    # ----------------------------------------

    ac_norm = sc_ac.transform(ac_raw)
    art_norm = sc_art.transform(art_raw)

    # Dataset for p(ART | AC)
    dataset = AXDataset(ac_norm, art_norm)

    train_loader = DataLoader(
        Subset(dataset, train_inner),
        batch_size=64,
        shuffle=True
    )

    val_loader = DataLoader(
        Subset(dataset, val_inner),
        batch_size=64,
        shuffle=False
    )

    # ----------------------------------------
    # 4. Train MDN
    # ----------------------------------------

    model = train_model(
        train_loader,
        val_loader,
        x_dim=ac_raw.shape[1],
        y_dim=art_raw.shape[1],
    )

    # ----------------------------------------
    # 5. Evaluate outer held-out fold
    # ----------------------------------------

    model.eval()

    with torch.no_grad():

        ac_fold = torch.from_numpy(
            ac_norm[test_idx]
        ).float().to(device)

        art_fold = torch.from_numpy(
            art_norm[test_idx]
        ).float().to(device)

        logp_standardized = model.log_prob(
            ac_fold,
            art_fold
        ).cpu().numpy()

    # ----------------------------------------
    # 6. Convert density back to original
    #    ART coordinate system
    # ----------------------------------------

    log_jacobian = np.log(sc_art.scale_).sum()

    logp_original = logp_standardized - log_jacobian

    logp_full[test_idx] = logp_original

phonemes_arr = np.array(phonemes)

H_art_given_ac_mdn = {}

for ph in np.unique(phonemes_arr):

    idx = np.where(phonemes_arr == ph)[0]

    if len(idx) < 30:
        continue

    H_art_given_ac_mdn[ph] = -logp_full[idx].mean()


   FOLD 0
Epoch 1: train=74.2510, val=71.1300
   ✓ New best model
Epoch 2: train=69.4307, val=69.5516
   ✓ New best model
Epoch 3: train=67.6451, val=68.8560
   ✓ New best model
Epoch 4: train=66.3445, val=68.4513
   ✓ New best model
Epoch 5: train=65.3719, val=67.9658
   ✓ New best model
Epoch 6: train=64.5519, val=68.0241
   No improvement (1 epochs)
Epoch 7: train=63.8437, val=68.1641
   No improvement (2 epochs)
Epoch 8: train=63.2337, val=68.2161
   No improvement (3 epochs)
Epoch 9: train=62.6432, val=68.3521
   No improvement (4 epochs)
Epoch 10: train=62.1002, val=68.3704
   No improvement (5 epochs)
Epoch 11: train=61.6054, val=68.6086
   No improvement (6 epochs)
Epoch 12: train=61.1754, val=68.7801
   No improvement (7 epochs)
Epoch 13: train=60.7601, val=68.7850
   No improvement (8 epochs)
Epoch 14: train=60.3765, val=69.2790
   No improvement (9 epochs)
Epoch 15: train=60.0260, val=69.8130
   No improvement (10 epochs)
Early stopping. Best val=67.9658

   FOLD 1
Epoch 1:

In [13]:
import pickle
with open("entropies/aeema_art_given_ac.pickle", "wb") as f:
    pickle.dump(H_art_given_ac_mdn, f)

In [14]:
logp_full = np.zeros(N)

for fold in range(k):

    print(f"\n==============================")
    print(f"   FOLD {fold}")
    print(f"==============================")

    test_idx = folds[fold]
    train_idx = np.hstack([
        folds[i] for i in range(k) if i != fold
    ])

    # ----------------------------------------
    # 1. Make internal train/validation split
    # ----------------------------------------

    train_idx_shuffled = train_idx.copy()
    np.random.shuffle(train_idx_shuffled)

    split = int(0.9 * len(train_idx_shuffled))

    train_inner = train_idx_shuffled[:split]
    val_inner = train_idx_shuffled[split:]

    # ----------------------------------------
    # 2. Fit scalers ONLY on outer training set
    # ----------------------------------------

    sc_art = StandardScaler()
    sc_ac = StandardScaler()

    sc_art.fit(art_raw[train_inner])
    sc_ac.fit(ac_raw[train_inner])

    # ----------------------------------------
    # 3. Transform all data using these scalers
    # ----------------------------------------

    art_norm = sc_art.transform(art_raw)
    ac_norm = sc_ac.transform(ac_raw)

    # Dataset for p(AC | ART)
    dataset = AXDataset(art_norm, ac_norm)

    train_loader = DataLoader(
        Subset(dataset, train_inner),
        batch_size=64,
        shuffle=True
    )

    val_loader = DataLoader(
        Subset(dataset, val_inner),
        batch_size=64,
        shuffle=False
    )

    # ----------------------------------------
    # 4. Train MDN
    # ----------------------------------------

    model = train_model(
        train_loader,
        val_loader,
        x_dim=art_raw.shape[1],
        y_dim=ac_raw.shape[1],
    )

    # ----------------------------------------
    # 5. Evaluate outer held-out fold
    # ----------------------------------------

    model.eval()

    with torch.no_grad():

        art_fold = torch.from_numpy(
            art_norm[test_idx]
        ).float().to(device)

        ac_fold = torch.from_numpy(
            ac_norm[test_idx]
        ).float().to(device)

        logp_standardized = model.log_prob(
            art_fold,
            ac_fold
        ).cpu().numpy()

    # ----------------------------------------
    # 6. Convert density back to original
    #    AC coordinate system
    # ----------------------------------------

    log_jacobian = np.log(sc_ac.scale_).sum()

    logp_original = logp_standardized - log_jacobian

    logp_full[test_idx] = logp_original

phonemes_arr = np.array(phonemes)

H_ac_given_art_mdn = {}

for ph in np.unique(phonemes_arr):

    idx = np.where(phonemes_arr == ph)[0]

    if len(idx) < 30:
        continue

    H_ac_given_art_mdn[ph] = -logp_full[idx].mean()


   FOLD 0
Epoch 1: train=81.6340, val=80.0785
   ✓ New best model
Epoch 2: train=78.7690, val=78.9620
   ✓ New best model
Epoch 3: train=77.6877, val=78.4903
   ✓ New best model
Epoch 4: train=76.9708, val=78.3882
   ✓ New best model
Epoch 5: train=76.3938, val=78.1498
   ✓ New best model
Epoch 6: train=75.9186, val=78.1213
   ✓ New best model
Epoch 7: train=75.4972, val=78.1118
   ✓ New best model
Epoch 8: train=75.1572, val=78.1788
   No improvement (1 epochs)
Epoch 9: train=74.8518, val=78.1326
   No improvement (2 epochs)
Epoch 10: train=74.5966, val=78.1422
   No improvement (3 epochs)
Epoch 11: train=74.3490, val=78.1248
   No improvement (4 epochs)
Epoch 12: train=74.1333, val=78.3755
   No improvement (5 epochs)
Epoch 13: train=73.9429, val=78.3110
   No improvement (6 epochs)
Epoch 14: train=73.7737, val=78.3510
   No improvement (7 epochs)
Epoch 15: train=73.5883, val=78.5229
   No improvement (8 epochs)
Epoch 16: train=73.4344, val=78.6114
   No improvement (9 epochs)
Epoch

In [15]:
with open("entropies/aeema_ac_given_art.pickle", "wb") as f:
    pickle.dump(H_ac_given_art_mdn, f)